In [1]:
import os
import sys
import json
import matplotlib.pyplot as plt
import pandas as pd


In [2]:


def read_json_file(file_path):
    """
    Reads a JSON file and returns its content.
    
    Args:
        file_path (str): The path to the JSON file.
        
    Returns:
        dict: The content of the JSON file.
    """

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"The file {file_path} does not exist.")
    
    with open(file_path, 'r') as file:
        return json.load(file)
    
def write_json_file(data, file_path):
    """
    Writes data to a JSON file.
    
    Args:
        data (dict): The data to be written to the file.
        file_path (str): The path to the JSON file.
    """
    if os.path.exists(file_path):
        print(f"Warning: The file {file_path} already exists and will be overwritten.", file=sys.stderr)
    
    with open(file_path, 'w') as file:
        json.dump(data, file, indent=4)


def write_json_str(data, file_path):
    """
    Writes data to a JSON file, converting non-serializable types (like sets).

    Args:
        data (dict): The data to be written to the file.
        file_path (str): The path to the JSON file.
    """
    def convert(obj):
        if isinstance(obj, set):
            return list(obj)
        raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")

    if os.path.exists(file_path):
        print(f"Warning: The file {file_path} already exists and will be overwritten.", file=sys.stderr)
    
    json_str = json.dumps(data, indent=4, default=convert)
    
    with open(file_path, 'w') as file:
        file.write(json_str)
        
def write_turtle_to_ttl(file_path, content):
    """
    Writes Turtle content to a TTL file.
    
    Args:
        file_path (str): The path to the TTL file.
        content (str): The Turtle content to be written.
    """
    content = "\n".join(content)  # Ensure content is a single string
    content = content.replace(' .', '.')
    content = content.replace(' ;', ';')

    with open(file_path, 'w') as file:
        file.write(content)

In [3]:
results_dir = "/Users/sefika/phd_projects/converse_relations/results/fewrel/"
experiments = ["original", "MT", "AI"]
input_files = ["report_head_tail_templates_with_desc_1.json", "report_head_tail_templates_without_desc_1.json",
               "report_tail_head_templates_with_desc_1.json", "report_tail_head_templates_without_desc_1.json"]

models = [{"model":"flan-t5-xl", "folder":"model_flan-t5-xl"},
          {"model":"Llama-3.1-8B-Instruct", "folder":"model_Llama-3.1-8B-Instruct"},
          {"model":"Qwen2.5-7B-Instruct", "folder": "model_Qwen2.5-7B-Instruct"},
          {"model":"Mistral-7B-Instruct-v0.3", "folder": "model_Mistral-7B-Instruct-v0.3"},
          {"model":"Qwen3-4B-Instruct-2507", "folder": "model_Qwen3-4B-Instruct-2507"}
          ]

In [4]:
results_all = []

In [11]:
for exp in experiments:
    for model in models:
        for input_file in input_files:
            report_file = f"{results_dir}{exp}/report_class/{model['folder']}/{input_file}"
            if not os.path.exists(report_file):
                print(f"Report file {report_file} does not exist. Skipping.")
                continue
            report_data = read_json_file(report_file)
            per_class_results = report_data['per_class']
            row_results = {"Experiment": exp, 
                           "Model": model['model'], 
                           "Input File": input_file,
                           "precision_other"  : per_class_results[""]['precision'],
                            "f1_other": per_class_results[""]['f1'],
                            "recall_other": per_class_results[""]['recall'],
                            "precision_child":per_class_results["child"]['precision'],
                            "f1_child": per_class_results["child"]['f1'],
                            "recall_child": per_class_results["child"]['recall'],
                            "precision_father": per_class_results["father"]['precision'],
                            "f1_father": per_class_results["father"]['f1'],
                            "recall_father": per_class_results["father"]['recall'],
                            "precision_followed_by": per_class_results["followed by"]['precision'],
                            "f1_followed_by": per_class_results["followed by"]['f1'],
                            "recall_followed_by": per_class_results["followed by"]['recall'],
                            "precision_follows": per_class_results["follows"]['precision'],
                            "f1_follows": per_class_results["follows"]['f1'],
                            "recall_follows": per_class_results["follows"]['recall'],
                            "precision_has_part": per_class_results["has part"]['precision'],
                            "f1_has_part": per_class_results["has part"]['f1'],
                            "recall_has_part": per_class_results["has part"]['recall'],
                            "precision_mother": per_class_results["mother"]['precision'],
                            "f1_mother": per_class_results["mother"]['f1'],
                            "recall_mother": per_class_results["mother"]['recall'],
                            "precision_part_of": per_class_results["part of"]['precision'],
                            "recall_part_of": per_class_results["part of"]['recall'],
                            "f1_part_of": per_class_results["part of"]['f1']
                           }
            results_all.append(row_results)
     

            

In [12]:
results_all_df = pd.DataFrame(results_all)
results_all_df.to_csv("results_all.csv", index=False)